# projet__python_DS
# Le prix de l'immobilier dépend-il de l'accès aux commerces alimentaires ?

*Projet de Python pour la data science (ENSAE, 2026) — [Prénom Nom], [Prénom Nom]*

Notebook 01_nettoyage_bpe

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import geopandas as gpd
from src import config
from src.telechargement import telecharger

In [2]:
colonnes_lues = [
    "DOM", "SDOM", "TYPEQU", "SIRET", "NOMRS", "APET",
    "DEPCOM", "DEP", "REG", "LIBCOM", "CODPOS",
    "LAMBERT_X", "LAMBERT_Y", "EPSG", "LONGITUDE", "LATITUDE",
    "QUALITE_XY", "QUALITE_GEOLOC",
    "DENS3", "DENS7",
]

chemin = telecharger(config.URL_BPE, "BPE25.parquet")
bpe = pd.read_parquet(chemin, columns=colonnes_lues)

BPE25.parquet : téléchargement depuis https://www.insee.fr/fr/statistiques/fichier/8217525/BPE25.parquet ...
BPE25.parquet : 172 Mo téléchargés


In [3]:
bpe = bpe.loc[bpe["SDOM"].isin(["B1", "B2"]) & (bpe["TYPEQU"] != "B103"), :].copy()

In [5]:
bpe_alim = bpe[colonnes_lues].copy()

for col in ["DOM", "SDOM", "TYPEQU", "SIRET", "DEPCOM", "DEP", "REG", "APET", "CODPOS", "DENS7"]:
    bpe_alim[col] = bpe_alim[col].astype("string")

for col in ["LAMBERT_X", "LAMBERT_Y", "LONGITUDE", "LATITUDE"]:
    bpe_alim[col] = pd.to_numeric(bpe_alim[col], errors="coerce")

bpe_alim = bpe_alim.drop_duplicates()
bpe_alim = bpe_alim.dropna(subset=["LONGITUDE", "LATITUDE"]).copy()

In [6]:
DROM = ["971", "972", "973", "974", "976"]
ABSENTS_DVF = ["57", "67", "68"]

bpe_alim = bpe_alim[~bpe_alim["DEP"].isin(DROM + ABSENTS_DVF)].copy()

In [7]:
bpe_geo = gpd.GeoDataFrame(
    bpe_alim,
    geometry=gpd.points_from_xy(bpe_alim["LONGITUDE"], bpe_alim["LATITUDE"]),
    crs="EPSG:4326",
)

bpe_geo = bpe_geo.to_crs("EPSG:2154")

Notebook 02_nettoyage_dvf

In [8]:
from src.densite import charger_grille, ajouter_densite

colonnes_dvf = [
    "id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere",
    "code_commune", "nom_commune", "code_departement",
    "type_local", "surface_reelle_bati", "surface_terrain",
    "longitude", "latitude",
]

chemin_dvf = telecharger(config.URL_DVF, "dvf_2025.csv.gz")
dvf_brut = pd.read_csv(chemin_dvf, dtype=str, usecols=colonnes_dvf, compression="gzip")

dvf_2025.csv.gz : téléchargement depuis https://www.data.gouv.fr/api/1/datasets/r/b3a0c6d2-2a5a-45a3-81a8-9e860caf5e14 ...
dvf_2025.csv.gz : 116 Mo téléchargés


In [9]:
dvf = dvf_brut[dvf_brut["type_local"].isin(["Maison", "Appartement"])].copy()

dvf = dvf.drop_duplicates(subset=["id_mutation", "type_local", "surface_reelle_bati"])

n_dvf = dvf.groupby("id_mutation").size()

ids_pro = dvf_brut.loc[dvf_brut["type_local"].str.startswith("Local", na=False), "id_mutation"].unique()

ids_un_logement = n_dvf[n_dvf == 1].index

dvf = dvf[dvf["id_mutation"].isin(ids_un_logement) & ~dvf["id_mutation"].isin(ids_pro)].copy()

In [10]:
colonnes_essentielles = [
    "valeur_fonciere", "surface_reelle_bati",
    "code_commune", "longitude", "latitude",
]

dvf = dvf.dropna(subset=colonnes_essentielles).copy()

for col in ["valeur_fonciere", "surface_reelle_bati", "surface_terrain", "longitude", "latitude"]:
    dvf[col] = pd.to_numeric(dvf[col], errors="coerce")

dans_metropole = (dvf["longitude"].between(-5.5, 10) & dvf["latitude"].between(41, 51.5))

dvf = dvf[dans_metropole].copy()
dvf = dvf[dvf["nature_mutation"] == "Vente"].copy()

In [11]:
dvf = dvf[(dvf["valeur_fonciere"] > 0) & (dvf["surface_reelle_bati"] >= 9)].copy()

dvf["prix_m2"] = dvf["valeur_fonciere"] / dvf["surface_reelle_bati"]

PRIX_M2_PLANCHER = 100

dvf = dvf[dvf["prix_m2"] >= PRIX_M2_PLANCHER].copy()

In [12]:
stats_communes = dvf.groupby(["code_commune", "nom_commune"])["prix_m2"].agg(mediane="median", nb_ventes="size")

communes_fiables = stats_communes[stats_communes["nb_ventes"] >= 100]

PRIX_M2_PLAFOND = 10 * communes_fiables["mediane"].max()

dvf = dvf[dvf["prix_m2"] <= PRIX_M2_PLAFOND].copy()

dvf["mediane_commune"] = dvf.groupby("code_commune")["prix_m2"].transform("median")
dvf["ratio"] = dvf["prix_m2"] / dvf["mediane_commune"]

dvf = dvf[dvf["ratio"].between(0.1, 10)].copy()
dvf = dvf.drop(columns=["mediane_commune", "ratio"])

In [13]:
grille = charger_grille(telecharger(config.URL_GRILLE, "grille_densite_2025.xlsx"))

dvf = ajouter_densite(dvf, grille)

grille_densite_2025.xlsx : téléchargement depuis https://www.insee.fr/fr/statistiques/fichier/8571524/fichier_diffusion_2025.xlsx ...
grille_densite_2025.xlsx : 2 Mo téléchargés
Ventes sans densité : 0


In [14]:
dvf_geo = gpd.GeoDataFrame(
    dvf,
    geometry=gpd.points_from_xy(dvf["longitude"], dvf["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:2154")

notebook 03_croisement

In [15]:
from scipy.spatial import cKDTree

bpe = bpe_geo.copy()
dvf = dvf_geo.copy()

In [16]:
codes_b2 = sorted(c for c in bpe["TYPEQU"].unique() if c.startswith("B2"))

types_commerces = {
    "hyper": ["B104"],
    "super": ["B105"],
    "proximite": ["B201", "B202"],
    "specialise": [c for c in codes_b2 if c not in ["B201", "B202"]],
}

categories = list(types_commerces)

In [17]:
for nom, codes in types_commerces.items():
    commerces = bpe.loc[bpe["TYPEQU"].isin(codes), ["geometry"]]

    plus_proche_type = gpd.sjoin_nearest(
        dvf[["geometry"]], commerces,
        how="left", distance_col=f"dist_{nom}",
    )
    plus_proche_type = plus_proche_type[~plus_proche_type.index.duplicated()]

    dvf[f"dist_{nom}"] = plus_proche_type[f"dist_{nom}"]

In [18]:
colonnes_dist = [f"dist_{nom}" for nom in categories]

dvf["dist_plus_proche"] = dvf[colonnes_dist].min(axis=1)
dvf["type_plus_proche"] = dvf[colonnes_dist].idxmin(axis=1).str.replace("dist_", "")

In [19]:
def calculer_rayons(q):
    return pd.DataFrame({
        cat: dvf.groupby("DENS7")[f"dist_{cat}"].quantile(q)
        for cat in categories
    })

rayons = calculer_rayons(0.99)

In [20]:
TAILLE_BLOC = 5_000

def indice_lineaire(xy_ventes, rayon_vente, arbre, rayons_commerces, comptage=False):
    resultat = np.zeros(len(xy_ventes))

    for debut in range(0, len(xy_ventes), TAILLE_BLOC):
        bloc = xy_ventes[debut:debut + TAILLE_BLOC]

        paires = cKDTree(bloc).sparse_distance_matrix(
            arbre, max_distance=rayon_vente, output_type="ndarray"
        )

        rayon_paire = np.minimum(
            rayon_vente, rayons_commerces[paires["j"]]
        )

        if comptage:
            contributions = (paires["v"] < rayon_paire).astype(int)
        else:
            contributions = np.clip(
                1 - paires["v"] / rayon_paire, 0, None
            )

        resultat[debut:debut + len(bloc)] = np.bincount(
            paires["i"],
            weights=contributions,
            minlength=len(bloc)
        )

    return resultat


def calculer_indices(df, rayons_q, comptage=False):
    resultats = pd.DataFrame(index=df.index)

    for cat in categories:
        commerces = bpe[bpe["TYPEQU"].isin(types_commerces[cat])]

        arbre = cKDTree(
            np.column_stack([commerces.geometry.x, commerces.geometry.y])
        )

        rayons_commerces = commerces["DENS7"].map(
            rayons_q[cat]
        ).to_numpy(dtype=float)

        colonne = f"{'nb_rayon' if comptage else 'acces'}_{cat}"
        resultats[colonne] = np.nan

        for niveau, rayon in rayons_q[cat].items():
            masque = (df["DENS7"] == niveau).to_numpy()

            if masque.sum() == 0:
                continue

            xy = np.column_stack([
                df.geometry.x[masque],
                df.geometry.y[masque]
            ])

            resultats.loc[masque, colonne] = indice_lineaire(
                xy, rayon, arbre, rayons_commerces,
                comptage=comptage
            )

    return resultats

In [22]:
import numpy as np
from scipy.spatial import cKDTree

In [23]:
indices = calculer_indices(dvf, rayons)
dvf[indices.columns] = indices

comptages = calculer_indices(dvf, rayons, comptage=True)
dvf[comptages.columns] = comptages

In [24]:
dvf_reg = dvf[dvf["prix_m2"] > 0].copy()

dvf_reg["log_prix_m2"] = np.log(dvf_reg["prix_m2"])

print("Observations :", len(dvf_reg))

Observations : 810517


On estime six régressions linéaires pour expliquer le logarithme du prix au m². Tous les modèles contrôlent la surface du logement, son type et son département. À partir de M2, on contrôle également la densité urbaine (DENS7).

Les six modèles étudient successivement :

M1–M2 : la distance au commerce le plus proche, sans puis avec contrôle de la densité.

M3 : l'interaction entre distance et densité.

M4 : les distances par catégorie de commerces.

M5 : le nombre de commerces accessibles (frontière nette).

M6 : l'accessibilité pondérée par la distance (frontière dégressive).

Les écarts-types sont robustes à l'hétéroscédasticité (HC3).

In [25]:
import statsmodels.formula.api as smf

dvf_reg["log_prix_m2"] = np.log(dvf_reg["prix_m2"])

In [26]:
controles = "surface_reelle_bati + C(type_local) + C(code_departement)"
densite = " + C(DENS7)"

formules = {
    "M1": "dist_plus_proche",
    "M2": "dist_plus_proche" + densite,
    "M3": "dist_plus_proche * C(DENS7)",
    "M4": "dist_hyper + dist_super + dist_proximite + dist_specialise" + densite,
    "M5": "nb_rayon_hyper + nb_rayon_super + nb_rayon_proximite + nb_rayon_specialise" + densite,
    "M6": "acces_hyper + acces_super + acces_proximite + acces_specialise" + densite,
}

modeles = {}

for nom, variables in formules.items():
    formule = f"log_prix_m2 ~ {variables} + {controles}"
    modeles[nom] = smf.ols(formule, data=dvf_reg).fit(cov_type="HC3")

In [27]:
for nom, modele in modeles.items():
    print(f"\n{nom} — R² = {modele.rsquared:.3f} — N = {int(modele.nobs)}")

    tableau = modele.summary2().tables[1]
    tableau = tableau.loc[
        ~tableau.index.str.contains(
            "code_departement|type_local|surface_reelle_bati|Intercept|DENS7\\[T\\."
        )
    ]

    display(tableau[["Coef.", "P>|z|"]].round(6))


M1 — R² = 0.493 — N = 810517


,Coef.,P>|z|
dist_plus_proche,-0.000068,0.0



M2 — R² = 0.509 — N = 810517


,Coef.,P>|z|
C(DENS7)[T.2],-0.111204,0.0
C(DENS7)[T.3],-0.096968,0.0
C(DENS7)[T.4],0.014579,0.0
C(DENS7)[T.5],-0.209930,0.0
C(DENS7)[T.6],-0.284730,0.0
C(DENS7)[T.7],-0.464222,0.0
dist_plus_proche,-0.000019,0.0



M3 — R² = 0.510 — N = 810517


,Coef.,P>|z|
C(DENS7)[T.2],-0.121216,0.000000
C(DENS7)[T.3],-0.083073,0.000000
C(DENS7)[T.4],0.040101,0.000000
C(DENS7)[T.5],-0.186354,0.000000
C(DENS7)[T.6],-0.211907,0.000000
C(DENS7)[T.7],-0.449121,0.000000
dist_plus_proche,0.000121,0.000000
dist_plus_proche:C(DENS7)[T.2],-0.000002,0.809335
dist_plus_proche:C(DENS7)[T.3],-0.000095,0.000000
dist_plus_proche:C(DENS7)[T.4],-0.000120,0.000000


... (2 outputs left out)


M5 — R² = 0.519 — N = 810517


,Coef.,P>|z|
C(DENS7)[T.2],-0.113007,0.0
C(DENS7)[T.3],-0.080345,0.0
C(DENS7)[T.4],0.039734,0.0
C(DENS7)[T.5],-0.186867,0.0
C(DENS7)[T.6],-0.275174,0.0
C(DENS7)[T.7],-0.457273,0.0
nb_rayon_hyper,0.013090,0.0
nb_rayon_super,0.004127,0.0
nb_rayon_proximite,-0.001913,0.0
nb_rayon_specialise,0.002053,0.0



M6 — R² = 0.517 — N = 810517


,Coef.,P>|z|
C(DENS7)[T.2],-0.140004,0.0
C(DENS7)[T.3],-0.106313,0.0
C(DENS7)[T.4],0.024233,0.0
C(DENS7)[T.5],-0.195787,0.0
C(DENS7)[T.6],-0.275419,0.0
C(DENS7)[T.7],-0.457180,0.0
acces_hyper,0.026725,0.0
acces_super,0.009351,0.0
acces_proximite,-0.004111,0.0
acces_specialise,0.004107,0.0


### Principaux résultats

**M1 — Distance au commerce le plus proche (R² = 0,493)**

La distance est négativement associée au prix immobilier : **100 mètres supplémentaires correspondent à environ −0,68 % du prix au m²** (p < 0,001).

**M2 — Ajout de la densité urbaine (R² = 0,509)**

Après contrôle de la densité, cette association diminue à **−0,19 % pour 100 mètres**. Une partie de la relation observée dans M1 s'explique donc par les différences entre territoires urbains et ruraux.

**M3 — Interaction entre distance et densité (R² = 0,510)**

L'association varie selon le territoire : elle est positive dans les grands centres urbains (environ **+1,2 % pour 100 mètres**) et négative dans les espaces ruraux les moins denses.

Une explication possible est qu'en ville, un léger éloignement des commerces peut correspondre à des quartiers résidentiels plus calmes, tandis qu'en zone rurale, la proximité des commerces représente un avantage.

**M4 — Distances par catégorie (R² = 0,512)**

Les distances aux quatre catégories sont négativement associées aux prix (p < 0,001). L'association est particulièrement marquée pour les **commerces de proximité (−1,7 % par km)** et les **supermarchés (−1,3 % par km)**.

**M5 — Frontière nette (R² = 0,519)**

Un nombre plus élevé d'hypermarchés, de supermarchés et de commerces spécialisés accessibles est associé à des prix plus élevés. En revanche, le nombre de commerces de proximité présente une association négative (**−0,19 % par commerce supplémentaire**).

Cela montre que la proximité d'un commerce et la concentration de nombreux commerces ne mesurent pas la même chose.

Une forte concentration de petits commerces pourrait s'accompagner de davantage de nuisances, tandis que la présence d'hypermarchés pourrait être associée à des territoires mieux équipés.

**M6 — Frontière dégressive (R² = 0,517)**

Les coefficients conservent les mêmes signes que dans M5. La pondération par la distance ne modifie donc pas les principaux résultats. La frontière nette présente un ajustement légèrement meilleur (**0,519 contre 0,517**).

### Conclusion des régressions

Les six modèles montrent que **l'accessibilité aux commerces alimentaires est statistiquement associée aux prix immobiliers**, mais que cette relation dépend de la densité urbaine et du type de commerce.

La proximité commerciale semble davantage valorisée dans les territoires ruraux, tandis que dans les grands centres urbains, un léger éloignement peut être associé à des prix plus élevés.

Enfin, la **frontière nette présente un ajustement légèrement meilleur** que la frontière dégressive (R² de 0,519 contre 0,517).

### Pour aller plus loin

Deux prolongements seraient intéressants :

- **Estimer séparément les catégories dans M5 et M6 :** les différentes catégories peuvent être corrélées en raison d'effets de regroupement clusters. Par exemple, un hypermarché est souvent entouré de commerces spécialisés. Estimer un modèle par catégorie permettrait d'étudier les résultats sans cette multicolinéarité, même si les effets des autres catégories ne seraient alors plus contrôlés.

- **Distinguer deux zones de distance :** dans les grands centres urbains, on pourrait compter séparément les commerces situés à moins de r mètres et ceux situés entre r et R mètres (R > r).
L'hypothèse serait que les commerces très proches diminuent les prix à cause des nuisances, tandis que ceux un peu plus éloignés les augmentent, car ils restent accessibles sans être gênants.

**Limite :** ces résultats restent des associations statistiques et ne permettent pas d'établir un lien de causalité.